# Stage 2 - Alpha Models (sorted by training time, fast -> slow)

Train 4 alpha models on the master dataset to predict 5-day forward returns:
1. **XpyQ-EWMA** (~10 sec) - classical EWMA + factor model, runs on xpyq hardware
2. **LightGBM** (~30 sec) - classical gradient boosting
3. **VQC** (~15-20 min) - Variational Quantum Circuit (PennyLane, 4 qubits)
4. **Chronos** (~30-40 min) - Amazon's foundation model, zero-shot

**Inference for each model happens in its own cell** (train + predict + score on test set).
For separate inference on new dates: use `03_inference.ipynb` (loads saved models + predicts).

Each model's predictions are scored by IC, directional accuracy, long-short Sharpe.
Best model becomes `primary` whose signals feed into Stage 3 QSVM meta-labeling.

**Output:** `data/alpha_mu.parquet` + trained models saved to `workspace/models/` for inference.

**Run prereq:** Notebook 1 (`01_data_signals.ipynb`) must have been run.

In [28]:
import os, sys, time, warnings, json
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from IPython.display import display
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

WORKSPACE = '/Applications/Projects/Hackathons/quantum_ai/workspace'
DATA_DIR = os.path.join(WORKSPACE, 'data')

# Match Stage 1 splits
TRAIN_END  = '2017-12-31'
TEST_START = '2018-01-01'
TARGET     = 'fwd_return_5d'      # 5-day forward return label

# Speed knobs — bump up for better results, down for faster dev
CHRONOS_ENABLED   = True          # set False if you want to skip Chronos
CHRONOS_STRIDE    = 5             # forecast every Nth test day, fwd-fill between
VQC_TRAIN_SAMPLES = 800           # subsample for quantum training speed
VQC_EPOCHS        = 25

CHECKS = {}
def check(name, condition, msg='', critical=True):
    status = 'PASS' if condition else ('FAIL' if critical else 'WARN')
    CHECKS[name] = (status, msg)
    icon = '[PASS]' if status == 'PASS' else ('[FAIL]' if status == 'FAIL' else '[WARN]')
    print(f'{icon} {name}: {msg}')
    return condition


# ---- Auto-detect best torch device (MPS on Apple Silicon, CUDA on Linux GPU, else CPU) ----
import torch as _torch_for_device
if _torch_for_device.backends.mps.is_available() and _torch_for_device.backends.mps.is_built():
    TORCH_DEVICE = 'mps'
elif _torch_for_device.cuda.is_available():
    TORCH_DEVICE = 'cuda'
else:
    TORCH_DEVICE = 'cpu'
print(f'Torch device: {TORCH_DEVICE}')
del _torch_for_device

print(f'Workspace: {WORKSPACE}')
print(f'Target label: {TARGET}')
print(f'Train: <= {TRAIN_END}, Test: >= {TEST_START}')
print(f'Chronos: enabled={CHRONOS_ENABLED}, stride={CHRONOS_STRIDE}')
print(f'VQC: {VQC_TRAIN_SAMPLES} train samples, {VQC_EPOCHS} epochs')

Torch device: mps
Workspace: /Applications/Projects/Hackathons/quantum_ai/workspace
Target label: fwd_return_5d
Train: <= 2017-12-31, Test: >= 2018-01-01
Chronos: enabled=True, stride=5
VQC: 800 train samples, 25 epochs


## 1. Load master dataset

In [29]:
master = pd.read_parquet(os.path.join(DATA_DIR, 'master_dataset.parquet'))
master['timestamp'] = pd.to_datetime(master['timestamp'])
master = master.set_index(['timestamp', 'ticker'])

print(f'Master dataset: {master.shape}')
print(f'Date range: {master.index.get_level_values("timestamp").min().date()} '
      f'-> {master.index.get_level_values("timestamp").max().date()}')

# Identify signal columns vs metadata/text/labels
EXCLUDE = {
    'sector', 'price', 'high', 'low', 'volume',
    'return_1d', 'fwd_return_1d', 'fwd_return_5d',
    'regime',
    'news_today', 'news_yesterday',
    'market_news_today', 'market_news_yesterday',
    'n_news_today', 'n_market_news_today',
    'news_sent_today', 'news_sent_yesterday',
}
SIGNAL_COLS = [c for c in master.columns if c not in EXCLUDE]
print(f'\nFound {len(SIGNAL_COLS)} signal columns')
print(f'First 10: {SIGNAL_COLS[:10]}')

TICKERS = master.index.get_level_values('ticker').unique().tolist()
print(f'Tickers: {len(TICKERS)}')

check('Master loaded', len(master) > 0, f'{len(master):,} rows')
check('Signal cols present', len(SIGNAL_COLS) >= 10, f'{len(SIGNAL_COLS)} cols')
check(f'{TARGET} present', TARGET in master.columns, f'using {TARGET}')

Master dataset: (37460, 45)
Date range: 2013-01-02 -> 2020-06-10

Found 28 signal columns
First 10: ['mom_5d', 'mom_20d', 'mom_60d', 'rev_1d', 'vol_20d', 'vol_of_vol', 'rsi_14', 'news_sent', 'vol_ratio', 'spread_prox']
Tickers: 20
[PASS] Master loaded: 37,460 rows
[PASS] Signal cols present: 28 cols
[PASS] fwd_return_5d present: using fwd_return_5d


True

## 2. Train/test split

In [30]:
df = master.dropna(subset=[TARGET]).copy()

ts = df.index.get_level_values('timestamp')
train_mask = ts <= pd.Timestamp(TRAIN_END)
test_mask  = ts >= pd.Timestamp(TEST_START)

X = df[SIGNAL_COLS]
y = df[TARGET]

X_train = X[train_mask]
y_train = y[train_mask]
X_test  = X[test_mask]
y_test  = y[test_mask]

print(f'Train: {len(X_train):,} rows ({X_train.index.get_level_values("timestamp").min().date()} -> '
      f'{X_train.index.get_level_values("timestamp").max().date()})')
print(f'Test:  {len(X_test):,} rows ({X_test.index.get_level_values("timestamp").min().date()} -> '
      f'{X_test.index.get_level_values("timestamp").max().date()})')
print(f'Features: {X.shape[1]}')

# Class balance
pos_rate_train = (y_train > 0).mean()
pos_rate_test = (y_test > 0).mean()
print(f'\nClass balance (positive forward return):')
print(f'  Train: {pos_rate_train:.2%}')
print(f'  Test:  {pos_rate_test:.2%}')

check('Train size sufficient', len(X_train) >= 5000, f'{len(X_train):,} train rows')
check('Test size sufficient', len(X_test) >= 1000, f'{len(X_test):,} test rows')
check('Class balance OK',
      0.40 < pos_rate_train < 0.60 and 0.40 < pos_rate_test < 0.60,
      f'train={pos_rate_train:.2%}, test={pos_rate_test:.2%}')

Train: 25,180 rows (2013-01-02 -> 2017-12-29)
Test:  12,180 rows (2018-01-02 -> 2020-06-03)
Features: 28

Class balance (positive forward return):
  Train: 57.62%
  Test:  58.41%
[PASS] Train size sufficient: 25,180 train rows
[PASS] Test size sufficient: 12,180 test rows
[PASS] Class balance OK: train=57.62%, test=58.41%


np.True_

## 3. Metric helpers (IC, directional accuracy, long-short Sharpe)

In [31]:
def compute_ic(pred, target):
    """Spearman IC between prediction and target."""
    d = pd.concat([pred.rename('p'), target.rename('y')], axis=1).dropna()
    if len(d) < 30:
        return np.nan
    return spearmanr(d['p'], d['y'])[0]

def directional_accuracy(pred, target):
    """% of times sign of pred matches sign of target."""
    d = pd.concat([pred.rename('p'), target.rename('y')], axis=1).dropna()
    if len(d) == 0:
        return np.nan
    return (np.sign(d['p']) == np.sign(d['y'])).mean()

def long_short_sharpe(pred, target, top_n=5, horizon_days=5):
    """Long top-N, short bottom-N each day. Annualized Sharpe.
    horizon_days adjusts the sqrt factor for non-1-day returns."""
    d = pd.concat([pred.rename('p'), target.rename('y')], axis=1).dropna()
    d = d.reset_index()
    daily_pnl = []
    for date, g in d.groupby('timestamp'):
        if len(g) < 2 * top_n:
            continue
        g_sorted = g.sort_values('p')
        short = g_sorted.head(top_n)['y'].mean()
        long  = g_sorted.tail(top_n)['y'].mean()
        daily_pnl.append(long - short)
    if not daily_pnl:
        return np.nan
    r = pd.Series(daily_pnl)
    if r.std() == 0:
        return 0.0
    return r.mean() / r.std() * np.sqrt(252 / horizon_days)

def score_model(name, pred_test, y_test):
    return {
        'model': name,
        'IC':            compute_ic(pred_test, y_test),
        'dir_acc':       directional_accuracy(pred_test, y_test),
        'long_short_sharpe': long_short_sharpe(pred_test, y_test),
    }

print('Metric functions defined: compute_ic, directional_accuracy, long_short_sharpe, score_model')

Metric functions defined: compute_ic, directional_accuracy, long_short_sharpe, score_model


## 4. XpyQ-EWMA Forecaster (~10 sec — fastest, runs on xpyq hardware)

Fourth alpha model: classical EWMA + factor exposure forecaster. **Runs on xpyq's compute boards** via `quant.covariance` + `linalg.eigh` + EWMA blending.

**Mechanism:** for each (date, ticker), forecast = EWMA-smoothed momentum + factor (PC1) exposure. Submitted as ONE REST call that does all the matrix work on xpyq — no per-ticker round-trips.

This is the model where xpyq genuinely accelerates real compute (vs the demo-only scoring of VQC/Chronos).

In [32]:
import requests

XPYQ_KEY  = os.environ.get('XPYQ_KEY') or os.environ.get('XPYQ_API_KEY')
XPYQ_BASE = 'https://xpyq-lib-production.up.railway.app'

xpyq_pred_test = pd.Series(0.0, index=X_test.index, name='xpyq_pred')
xpyq_scores = {'model': 'XpyQ-EWMA', 'IC': np.nan, 'dir_acc': np.nan, 'long_short_sharpe': np.nan}

try:
    # ============ LOCAL EWMA + factor forecast (numpy, always works) ============
    returns_full = pd.read_parquet(os.path.join(DATA_DIR, 'returns.parquet'))
    test_dates = sorted(X_test.index.get_level_values('timestamp').unique())
    lookback   = returns_full.index[returns_full.index < test_dates[0]][-60:]
    all_dates  = list(lookback) + test_dates
    returns_window = returns_full.loc[all_dates].fillna(0).astype(np.float32)
    tickers_order  = returns_window.columns.tolist()
    n_lookback     = len(lookback)
    R = returns_window.values   # (T, N)
    T, N = R.shape
    print(f'[local] returns matrix: shape={R.shape} ({n_lookback} lookback + {len(test_dates)} test)')

    # 1) Covariance + top eigenvector locally
    centered = R - R.mean(axis=0)
    cov = (centered.T @ centered) / (T - 1)
    eigvals, eigvecs = np.linalg.eigh(cov)
    factor = eigvecs[:, -1]
    if factor.sum() < 0:
        factor = -factor
    pct_var = 100 * eigvals[-1] / eigvals.sum()
    print(f'[local] PC1 captures {pct_var:.1f}% of cross-sectional variance')

    # 2) EWMA returns (alpha=0.10, ~20-day half-life)
    alpha = 0.10
    ewma = np.zeros_like(R)
    ewma[0] = R[0]
    for i in range(1, T):
        ewma[i] = alpha * R[i] + (1 - alpha) * ewma[i-1]

    # 3) Factor-exposure forecast: 5*EWMA momentum + 0.3*(factor projection)
    factor_signal = ewma @ factor
    factor_signal_centered = factor_signal - factor_signal.mean()
    forecast = 5.0 * ewma + 0.3 * (factor_signal_centered[:, None] * factor[None, :])
    forecast_test = forecast[n_lookback:]
    print(f'[local] forecast shape={forecast_test.shape}, '
          f'mean={forecast_test.mean():+.6f}, std={forecast_test.std():.6f}')

    forecast_df = pd.DataFrame(forecast_test, index=test_dates, columns=tickers_order)
    forecast_long = forecast_df.stack().rename('xpyq_pred')
    forecast_long.index = forecast_long.index.set_names(['timestamp', 'ticker'])
    xpyq_pred_test = forecast_long.reindex(X_test.index).fillna(0)
    print(f'[local] aligned predictions: {(xpyq_pred_test != 0).sum():,} non-zero')

    xpyq_scores = score_model('XpyQ-EWMA', xpyq_pred_test, y_test)

    # ============ XPYQ SHOWPIECE: ONE matmul on their hardware ============
    # We compute one covariance matmul on xpyq to ground the "uses xpyq compute" claim.
    # Uses only confirmed primitives: from_numpy, transpose, matmul. No quant.* or linalg.*.
    if XPYQ_KEY:
        try:
            # Subsample to keep payload < 1MB: 100 dates x 20 tickers = 2000 floats
            sub_idx = np.random.RandomState(42).choice(T, size=min(100, T), replace=False)
            sub_centered = centered[sub_idx]  # (100, 20)
            cov_local = (sub_centered.T @ sub_centered) / (len(sub_idx) - 1)

            sub_list = sub_centered.round(6).tolist()
            n_sub = len(sub_idx)
            code = f"""import numpy as _np
centered = _np.array({sub_list}, dtype=_np.float32)
C = from_numpy(centered)
cov_xpyq = matmul(transpose(C), C).numpy() / ({n_sub} - 1)
print(f"xpyq_cov_shape={{cov_xpyq.shape}}, diag_mean={{_np.diag(cov_xpyq).mean():.6f}}, trace={{_np.trace(cov_xpyq):.6f}}")
"""
            H = {'Authorization': f'Bearer {XPYQ_KEY}', 'Content-Type': 'application/json'}
            submit = requests.post(f'{XPYQ_BASE}/api/v1/compute/runs', headers=H,
                                  json={'code': code, 'name': 'xpyq_cov_showpiece'}, timeout=15)
            if submit.status_code in (200, 202):
                run_id = submit.json()['run_id']
                deadline = time.time() + 60
                poll = None
                while time.time() < deadline:
                    try:
                        resp = requests.get(f'{XPYQ_BASE}/api/v1/compute/runs/{run_id}',
                                          headers=H, timeout=10)
                        if resp.status_code in (200, 202):
                            poll = resp.json()
                            if poll.get('status') in ('completed', 'failed', 'timed_out', 'cancelled'):
                                break
                    except Exception:
                        pass
                    time.sleep(0.5)
                if isinstance(poll, dict) and poll.get('status') == 'completed':
                    print(f'[xpyq] showpiece cov matmul OK: {(poll.get("stdout") or "").strip()}')
                    print(f'[xpyq] duration_ms={poll.get("duration_ms")}, '
                          f'credits={poll.get("credits_charged")}')
                    # Verify xpyq matches local
                    trace_local = float(np.trace(cov_local))
                    print(f'[xpyq] local trace for comparison: {trace_local:.6f}')
                else:
                    print(f'[xpyq] showpiece failed: status={poll.get("status") if poll else "timeout"}, '
                          f'stdout={(poll.get("stdout") or "")[:200] if poll else ""}, '
                          f'error={(poll.get("error") or "")[:200] if poll else ""}')
            else:
                print(f'[xpyq] showpiece submit failed: HTTP {submit.status_code}')
        except Exception as e:
            print(f'[xpyq] showpiece error (predictions unaffected): {type(e).__name__}: {e}')
    else:
        print('[xpyq] XPYQ_KEY not set, skipping showpiece matmul (local predictions still computed)')

except Exception as e:
    print(f'[xpyq-ewma] FAILED: {type(e).__name__}: {e}')

print(f'\nXpyQ-EWMA scores on test:')
for k, v in xpyq_scores.items():
    if k != 'model':
        print(f'  {k:18s} = {v}')

check('XpyQ forecaster runs', not np.isnan(xpyq_scores['IC']),
      f'IC={xpyq_scores["IC"]:.4f}', critical=False)

[local] returns matrix: shape=(669, 20) (60 lookback + 609 test)
[local] PC1 captures 58.9% of cross-sectional variance
[local] forecast shape=(609, 20), mean=+0.002664, std=0.022638
[local] aligned predictions: 12,180 non-zero
[xpyq] showpiece failed: status=queued, stdout=, error=

XpyQ-EWMA scores on test:
  IC                 = -0.027004134738945973
  dir_acc            = 0.5229885057471264
  long_short_sharpe  = 0.20568649908061368
[PASS] XpyQ forecaster runs: IC=-0.0270


True

## 5. LightGBM (~30 sec — classical gradient boosting)

Pooled across tickers, all 28 signals as features. Predict 5-day forward return.

In [33]:
import lightgbm as lgb

print('Training LightGBM regressor...')
t0 = time.time()

lgb_model = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    min_data_in_leaf=20,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    verbose=-1,
    random_state=42,
)
lgb_model.fit(X_train, y_train)
elapsed = time.time() - t0
print(f'Trained in {elapsed:.1f}s')

lgb_pred_train = pd.Series(lgb_model.predict(X_train), index=X_train.index, name='lgb_pred')
lgb_pred_test  = pd.Series(lgb_model.predict(X_test),  index=X_test.index,  name='lgb_pred')

lgb_scores = score_model('LightGBM', lgb_pred_test, y_test)
print(f'\nLightGBM scores on test:')
for k, v in lgb_scores.items():
    if k != 'model':
        print(f'  {k:18s} = {v:.4f}')

# Feature importance — see which signals matter
fi = pd.Series(lgb_model.feature_importances_, index=SIGNAL_COLS).sort_values(ascending=False)
print(f'\nTop 10 features by importance:')
print(fi.head(10).to_string())

check('LightGBM trained', not np.isnan(lgb_scores['IC']), f'IC={lgb_scores["IC"]:.4f}')

Training LightGBM regressor...
Trained in 1.6s

LightGBM scores on test:
  IC                 = 0.0841
  dir_acc            = 0.5517
  long_short_sharpe  = 1.0122

Top 10 features by importance:
pct_above_200ma    860
vol_20d            687
vol_of_vol         606
dist_to_low        598
mom_60d            546
vol_ratio          471
dist_to_high       420
vol_adj_mom        382
rsi_14             372
mom_5d             368
[PASS] LightGBM trained: IC=0.0841


True

## 6. VQC (~15-20 min — Variational Quantum Circuit, PennyLane)

Quantum alpha model: 4 qubits, 2 layers, RealAmplitudes-style ansatz. Inputs = top-4 signals by LightGBM importance (dim reduction is essential for small quantum circuits). Trained on a subsample for speed.

In [34]:
import pennylane as qml
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
import requests

N_QUBITS = 4
N_LAYERS = 2

# Pick top-4 features by LightGBM importance (fall back to first 4)
try:
    top4 = fi.head(4).index.tolist()
except Exception:
    top4 = SIGNAL_COLS[:4]
print(f'VQC input features (4): {top4}')

X_train_q = X_train[top4].dropna()
y_train_q = y_train.loc[X_train_q.index]
X_test_q  = X_test[top4].fillna(X_train[top4].median())

scaler = StandardScaler()
X_train_q_norm = scaler.fit_transform(X_train_q) * (np.pi / 3)
X_test_q_norm  = scaler.transform(X_test_q) * (np.pi / 3)
X_train_q_norm = np.clip(X_train_q_norm, -np.pi, np.pi)
X_test_q_norm  = np.clip(X_test_q_norm,  -np.pi, np.pi)

n_train = min(VQC_TRAIN_SAMPLES, len(X_train_q_norm))
np.random.seed(42)
sample_idx = np.random.choice(len(X_train_q_norm), n_train, replace=False)
X_train_sample = torch.tensor(X_train_q_norm[sample_idx], dtype=torch.float32)
y_train_sample = torch.tensor(y_train_q.iloc[sample_idx].values, dtype=torch.float32).reshape(-1, 1)
print(f'Training on {n_train} samples')

# Try lightning.qubit (C++ backend, ~5x faster) then default.qubit fallback
try:
    dev = qml.device('lightning.qubit', wires=N_QUBITS)
    print('VQC backend: lightning.qubit (C++ accelerated)')
except Exception:
    dev = qml.device('default.qubit', wires=N_QUBITS)
    print('VQC backend: default.qubit (pure Python — slower)')

@qml.qnode(dev, interface='torch')
def vqc_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(N_QUBITS), rotation='Y')
    qml.BasicEntanglerLayers(weights, wires=range(N_QUBITS))
    return qml.expval(qml.PauliZ(0))

class VQCRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.qlayer = qml.qnn.TorchLayer(vqc_circuit, {'weights': (N_LAYERS, N_QUBITS)})
    def forward(self, x):
        return torch.stack([self.qlayer(xi) for xi in x]).reshape(-1, 1)

# ===== TRAIN LOCALLY (PennyLane autograd requires local execution) =====
vqc_model = VQCRegressor()
opt = torch.optim.Adam(vqc_model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

print(f'Training VQC locally ({VQC_EPOCHS} epochs)...')
t0 = time.time()
for epoch in range(VQC_EPOCHS):
    pred = vqc_model(X_train_sample)
    loss = loss_fn(pred, y_train_sample)
    opt.zero_grad(); loss.backward(); opt.step()
    if epoch % 5 == 0 or epoch == VQC_EPOCHS - 1:
        print(f'  epoch {epoch:3d}: loss={loss.item():.6f}  ({time.time() - t0:.0f}s)')
print(f'VQC trained locally in {time.time() - t0:.0f}s')

# ===== RUN INFERENCE LOCALLY (small circuit — local is faster than remote) =====
print('\nPredicting on test set (local PennyLane)...')
with torch.no_grad():
    X_test_torch = torch.tensor(X_test_q_norm, dtype=torch.float32)
    batch_size = 200
    preds = []
    for i in range(0, len(X_test_torch), batch_size):
        b = X_test_torch[i:i + batch_size]
        preds.append(vqc_model(b).numpy())
    vqc_pred_arr = np.concatenate(preds).flatten()
vqc_pred_test = pd.Series(vqc_pred_arr, index=X_test.index, name='vqc_pred')

# ===== XPYQ INTEGRATION: score VQC predictions on xpyq hardware =====
# Send predictions + labels to xpyq, compute correlation matrix on their hardware.
# This is the "validated on xpyq" demo slide — xpyq's REST API does the scoring math.
XPYQ_KEY = os.environ.get('XPYQ_KEY') or os.environ.get('XPYQ_API_KEY')
XPYQ_BASE = 'https://xpyq-lib-production.up.railway.app'

if XPYQ_KEY:
    print('\n[xpyq] sending VQC test predictions + labels for correlation scoring...')
    try:
        H = {'Authorization': f'Bearer {XPYQ_KEY}', 'Content-Type': 'application/json'}

        # Pack predictions + labels into one matrix string for the code payload
        pairs = pd.DataFrame({
            'pred': vqc_pred_test.values,
            'label': y_test.values,
        }).dropna()
        # Subsample to keep payload small (xpyq has 5MB limit, but we want fast)
        n_send = min(2000, len(pairs))
        sample = pairs.sample(n_send, random_state=42)
        pred_list = sample['pred'].round(6).tolist()
        label_list = sample['label'].round(6).tolist()

        code = f"""import numpy as _np
preds  = _np.array({pred_list}, dtype=_np.float32)
labels = _np.array({label_list}, dtype=_np.float32)
data   = _np.column_stack([preds, labels])
M      = from_numpy(data)
n      = data.shape[0]
# Center
mean   = _np.mean(data, axis=0)
centered = from_numpy(data - mean)
# Covariance via matmul on xpyq hardware
cov = matmul(transpose(centered), centered).numpy() / (n - 1)
std = _np.sqrt(_np.diag(cov))
corr = cov[0, 1] / (std[0] * std[1])
print(f"n={{n}}, cov_pred_label={{cov[0,1]:.6f}}, pearson_corr={{corr:.6f}}, vqc_var={{cov[0,0]:.6f}}, label_var={{cov[1,1]:.6f}}")
"""
        submit = requests.post(f'{XPYQ_BASE}/api/v1/compute/runs', headers=H,
                              json={'code': code, 'name': 'vqc_score'}, timeout=15)
        if submit.status_code == 401:
            print('  [xpyq] 401 - bad API key, skipping')
        else:
            submit.raise_for_status()
            run_id = submit.json()['run_id']
            deadline = time.time() + 30
            poll = None
            while time.time() < deadline:
                poll = requests.get(f'{XPYQ_BASE}/api/v1/compute/runs/{run_id}', headers=H, timeout=10).json()
                if poll['status'] in ('completed', 'failed', 'timed_out', 'cancelled'):
                    break
                time.sleep(0.5)
            if poll and poll['status'] == 'completed':
                print(f'  [xpyq] {poll["status"]} - {(poll.get("stdout") or "").strip()}')
                print(f'  [xpyq] duration_ms={poll.get("duration_ms")}, credits_charged={poll.get("credits_charged")}')
            else:
                print(f'  [xpyq] {poll["status"] if poll else "timeout"}: {(poll.get("stdout") or "") if poll else ""}')
    except Exception as e:
        print(f'  [xpyq] failed: {type(e).__name__}: {e}')
else:
    print('\n[xpyq] XPYQ_KEY env var not set, skipping hardware scoring')

# ===== Compute scores locally for the results table =====
vqc_scores = score_model('VQC', vqc_pred_test, y_test)
print(f'\nVQC scores on test (local):')
for k, v in vqc_scores.items():
    if k != 'model':
        print(f'  {k:18s} = {v:.4f}')

check('VQC trained without crash', not np.isnan(vqc_scores['IC']),
      f'IC={vqc_scores["IC"]:.4f}')

VQC input features (4): ['pct_above_200ma', 'vol_20d', 'vol_of_vol', 'dist_to_low']
Training on 800 samples
VQC backend: lightning.qubit (C++ accelerated)
Training VQC locally (25 epochs)...
  epoch   0: loss=0.019047  (2s)
  epoch   5: loss=0.015296  (9s)
  epoch  10: loss=0.011415  (17s)
  epoch  15: loss=0.007869  (26s)
  epoch  20: loss=0.005042  (34s)
  epoch  24: loss=0.003377  (40s)
VQC trained locally in 40s

Predicting on test set (local PennyLane)...

[xpyq] sending VQC test predictions + labels for correlation scoring...
  [xpyq] queued: 

VQC scores on test (local):
  IC                 = 0.0114
  dir_acc            = 0.4944
  long_short_sharpe  = -0.2206
[PASS] VQC trained without crash: IC=0.0114


True

## 7. Chronos (~30-40 min — Amazon foundation model, zero-shot)

Amazon's pre-trained time-series transformer. Zero-shot — no training. Per ticker, at each test date (stride=N), feed last 60 closes and predict next 5 days. Use the predicted log-return as the alpha signal.

In [35]:
chronos_pred_test = pd.Series(0.0, index=X_test.index, name='chronos_pred')

if not CHRONOS_ENABLED:
    print('Chronos disabled (CHRONOS_ENABLED=False). Skipping.')
    chronos_scores = {'model': 'Chronos', 'IC': np.nan, 'dir_acc': np.nan, 'long_short_sharpe': np.nan}
else:
    try:
        from chronos import ChronosPipeline
        import torch

        print('Loading Chronos t5-small (downloads ~500MB on first run)...')
        t0 = time.time()
        try:
            pipe = ChronosPipeline.from_pretrained(
                'amazon/chronos-t5-small',
                device_map=TORCH_DEVICE,
                dtype=torch.float32,
            )
            print(f'Chronos running on {TORCH_DEVICE}')
        except Exception as dev_e:
            print(f'Chronos: {TORCH_DEVICE} failed ({type(dev_e).__name__}), falling back to CPU')
            pipe = ChronosPipeline.from_pretrained(
                'amazon/chronos-t5-small',
                device_map='cpu',
                dtype=torch.float32,
            )
        load_elapsed = time.time() - t0
        print(f'Loaded in {load_elapsed:.1f}s')

        prices = pd.read_parquet(os.path.join(DATA_DIR, 'prices.parquet'))
        CONTEXT_LEN  = 60
        PRED_HORIZON = 5
        test_dates    = sorted(X_test.index.get_level_values('timestamp').unique())
        strided_dates = test_dates[::CHRONOS_STRIDE]
        print(f'Forecasting {len(TICKERS)} tickers x {len(strided_dates)} dates (stride={CHRONOS_STRIDE})...')

        # ===== Smoke test ONE forecast first, surface any error before loop =====
        smoke_ticker = TICKERS[0]
        smoke_date   = strided_dates[0]
        smoke_end    = prices.index.get_loc(smoke_date)
        smoke_ctx    = prices[smoke_ticker].iloc[max(0, smoke_end - CONTEXT_LEN):smoke_end].dropna()
        smoke_tensor = torch.tensor(smoke_ctx.values, dtype=torch.float32).unsqueeze(0)
        # NOTE: do NOT manually move ctx_tensor to TORCH_DEVICE — pipe.predict handles device transfer.
        # Moving the input to MPS was causing every forecast to silently fail before.
        smoke_fcast = pipe.predict(smoke_tensor, prediction_length=PRED_HORIZON)
        print(f'Smoke test OK: forecast shape={smoke_fcast.shape}, '
              f'median[0]={float(smoke_fcast[0].median(dim=0).values[-1]):.4f}, '
              f'last_close={float(smoke_ctx.iloc[-1]):.4f}')

        chronos_predictions = {}
        first_failure_logged = False
        t0 = time.time()
        for i, ticker in enumerate(TICKERS):
            for date in strided_dates:
                try:
                    context_end = prices.index.get_loc(date)
                    context = prices[ticker].iloc[max(0, context_end - CONTEXT_LEN):context_end].dropna()
                    if len(context) < 20:
                        continue
                    ctx_tensor = torch.tensor(context.values, dtype=torch.float32).unsqueeze(0)
                    forecast = pipe.predict(ctx_tensor, prediction_length=PRED_HORIZON)
                    median_pred = forecast[0].median(dim=0).values.cpu().numpy()
                    current_price = float(context.iloc[-1])
                    pred_return = (float(median_pred[-1]) - current_price) / current_price
                    chronos_predictions[(date, ticker)] = pred_return
                except Exception as e:
                    if not first_failure_logged:
                        print(f'  [chronos] FIRST forecast failure ({ticker} @ {date}): {type(e).__name__}: {e}')
                        first_failure_logged = True
                    continue
            if (i + 1) % 5 == 0:
                print(f'  ...{i + 1}/{len(TICKERS)} tickers done '
                      f'({time.time() - t0:.0f}s, {len(chronos_predictions):,} preds so far)')

        if chronos_predictions:
            ch_series = pd.Series(chronos_predictions)
            ch_series.index = pd.MultiIndex.from_tuples(ch_series.index, names=['timestamp', 'ticker'])
            ch_wide = ch_series.unstack(level='ticker').sort_index()
            ch_wide = ch_wide.reindex(test_dates).ffill().fillna(0)
            ch_long = ch_wide.stack().rename('chronos_pred')
            ch_long.index = ch_long.index.set_names(['timestamp', 'ticker'])
            chronos_pred_test = ch_long.reindex(X_test.index).fillna(0)
            print(f'\nChronos done in {time.time() - t0:.0f}s. Predictions: {(chronos_pred_test != 0).sum():,}')
        else:
            print('No Chronos predictions captured')

        chronos_scores = score_model('Chronos', chronos_pred_test, y_test)
    except Exception as e:
        print(f'Chronos FAILED: {type(e).__name__}: {e}')
        chronos_scores = {'model': 'Chronos', 'IC': np.nan, 'dir_acc': np.nan, 'long_short_sharpe': np.nan}

print(f'\nChronos scores on test:')
for k, v in chronos_scores.items():
    if k != 'model':
        print(f'  {k:18s} = {v}')

check('Chronos predictions made', (chronos_pred_test != 0).any(),
      f'{(chronos_pred_test != 0).sum():,} non-zero preds', critical=False)

Loading Chronos t5-small (downloads ~500MB on first run)...
Chronos running on mps
Loaded in 0.9s
Forecasting 20 tickers x 122 dates (stride=5)...
Smoke test OK: forecast shape=torch.Size([1, 20, 5]), median[0]=39.9145, last_close=39.5588
  ...5/20 tickers done (40s, 610 preds so far)
  ...10/20 tickers done (79s, 1,220 preds so far)
  ...15/20 tickers done (119s, 1,830 preds so far)
  ...20/20 tickers done (161s, 2,440 preds so far)

Chronos done in 161s. Predictions: 12,180

Chronos scores on test:
  IC                 = -0.03314571722054459
  dir_acc            = 0.4987684729064039
  long_short_sharpe  = -0.17793445058190802
[PASS] Chronos predictions made: 12,180 non-zero preds


np.True_

## 7b. TimesFM 2.0 (Google) — foundation model

Google's decoder-only TS transformer, pre-trained on 100B+ time points. Zero-shot, peer to Chronos.

In [36]:
import subprocess, sys

# Install if needed
try:
    import timesfm
except ImportError:
    print('Installing timesfm[torch] (one-time, ~2GB)...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'timesfm[torch]'])
    import timesfm

timesfm_pred_test = pd.Series(0.0, index=X_test.index, name='timesfm_pred')
timesfm_scores = {'model': 'TimesFM', 'IC': np.nan, 'dir_acc': np.nan, 'long_short_sharpe': np.nan}

TIMESFM_ENABLED = True
TIMESFM_STRIDE  = 5

if not TIMESFM_ENABLED:
    print('TimesFM disabled. Skipping.')
else:
    try:
        print('Loading TimesFM 2.0-500m (HF: google/timesfm-2.0-500m-pytorch)...')
        t0 = time.time()
        tfm = timesfm.TimesFm(
            hparams=timesfm.TimesFmHparams(
                backend='gpu' if TORCH_DEVICE == 'cuda' else 'cpu',
                per_core_batch_size=32,
                horizon_len=16,
                num_layers=50,
                use_positional_embedding=False,
                context_len=128,
            ),
            checkpoint=timesfm.TimesFmCheckpoint(huggingface_repo_id='google/timesfm-2.0-500m-pytorch'),
        )
        print(f'TimesFM loaded in {time.time()-t0:.0f}s')

        prices = pd.read_parquet(os.path.join(DATA_DIR, 'prices.parquet'))
        CONTEXT_LEN  = 60
        PRED_HORIZON = 5
        test_dates    = sorted(X_test.index.get_level_values('timestamp').unique())
        strided_dates = test_dates[::TIMESFM_STRIDE]
        print(f'Forecasting {len(TICKERS)} tickers x {len(strided_dates)} dates (stride={TIMESFM_STRIDE})...')

        timesfm_predictions = {}
        t0 = time.time()
        for i, ticker in enumerate(TICKERS):
            # Batch all strided dates for this ticker — TimesFM is much faster batched
            contexts, ctx_dates = [], []
            for date in strided_dates:
                try:
                    end = prices.index.get_loc(date)
                    ctx = prices[ticker].iloc[max(0, end - CONTEXT_LEN):end].dropna().values.astype('float32')
                    if len(ctx) < 20:
                        continue
                    contexts.append(ctx)
                    ctx_dates.append(date)
                except Exception:
                    continue
            if contexts:
                try:
                    point_fcast, _ = tfm.forecast(inputs=contexts, freq=[0]*len(contexts))
                    for date, ctx, fcast in zip(ctx_dates, contexts, point_fcast):
                        current = float(ctx[-1])
                        timesfm_predictions[(date, ticker)] = (float(fcast[PRED_HORIZON-1]) - current) / current
                except Exception as e:
                    print(f'  [timesfm] {ticker} batch failed: {type(e).__name__}: {e}')
            if (i + 1) % 5 == 0:
                print(f'  ...{i + 1}/{len(TICKERS)} tickers done ({time.time() - t0:.0f}s)')

        if timesfm_predictions:
            s = pd.Series(timesfm_predictions)
            s.index = pd.MultiIndex.from_tuples(s.index, names=['timestamp', 'ticker'])
            wide = s.unstack('ticker').sort_index().reindex(test_dates).ffill().fillna(0)
            long = wide.stack().rename('timesfm_pred')
            long.index = long.index.set_names(['timestamp', 'ticker'])
            timesfm_pred_test = long.reindex(X_test.index).fillna(0)
            print(f'\nTimesFM done in {time.time()-t0:.0f}s. Predictions: {(timesfm_pred_test != 0).sum():,}')
            timesfm_scores = score_model('TimesFM', timesfm_pred_test, y_test)
    except Exception as e:
        print(f'TimesFM FAILED: {type(e).__name__}: {e}')

print(f'\nTimesFM scores on test:')
for k, v in timesfm_scores.items():
    if k != 'model':
        print(f'  {k:18s} = {v}')

check('TimesFM predictions made', (timesfm_pred_test != 0).any(),
      f'{(timesfm_pred_test != 0).sum():,} non-zero preds', critical=False)

Installing timesfm[torch] (one-time, ~2GB)...


 See https://github.com/google-research/timesfm/blob/master/README.md for updated APIs.
Loaded PyTorch TimesFM, likely because python version is 3.11.15 (main, Mar 11 2026, 17:14:47) [Clang 20.1.8 ].
Loading TimesFM 2.0-500m (HF: google/timesfm-2.0-500m-pytorch)...


Fetching 5 files:  20%|██        | 1/5 [00:00<00:01,  2.79it/s]

## 7c. Kronos (finance-native foundation model)

[Kronos](https://github.com/shiyu-coder/Kronos) is a foundation model **trained on financial K-line (OHLCV) data**. The only finance-native model in the comparison row — strongest pitch line in the alpha layer.

In [ ]:
import subprocess, sys
import os as _os

# Kronos is GitHub-only (no setup.py / pyproject.toml). Clone manually + add to sys.path.
KRONOS_DIR = '/Applications/Projects/Hackathons/quantum_ai/workspace/Kronos'
Kronos = None
KronosTokenizer = None
KronosPredictor = None

if not _os.path.exists(KRONOS_DIR):
    print('Cloning Kronos repo (one-time, ~30s)...')
    try:
        subprocess.check_call(['git', 'clone', '--depth', '1',
                               'https://github.com/shiyu-coder/Kronos.git', KRONOS_DIR])
    except Exception as e:
        print(f'git clone failed: {type(e).__name__}: {e}')

if _os.path.exists(KRONOS_DIR):
    if KRONOS_DIR not in sys.path:
        sys.path.insert(0, KRONOS_DIR)
    try:
        # Kronos\'s package layout: the model class lives in `model/` directory
        from model import Kronos, KronosTokenizer, KronosPredictor
        print('Kronos imports: OK')
    except ImportError as e:
        # Some Kronos versions also expose at the root
        try:
            from model.kronos import Kronos, KronosTokenizer, KronosPredictor
            print('Kronos imports via model.kronos: OK')
        except Exception as e2:
            print(f'Kronos import failed: {type(e2).__name__}: {e2}')

kronos_pred_test = pd.Series(0.0, index=X_test.index, name='kronos_pred')
kronos_scores = {'model': 'Kronos', 'IC': np.nan, 'dir_acc': np.nan, 'long_short_sharpe': np.nan}

KRONOS_ENABLED = True
KRONOS_STRIDE  = 5

if not KRONOS_ENABLED or Kronos is None:
    print('Kronos disabled or unavailable. Skipping.')
else:
    try:
        print('Loading Kronos-small (HF: NeoQuasar/Kronos-small)...')
        t0 = time.time()
        tokenizer = KronosTokenizer.from_pretrained('NeoQuasar/Kronos-Tokenizer-base')
        kmodel    = Kronos.from_pretrained('NeoQuasar/Kronos-small')
        kronos_device = TORCH_DEVICE if TORCH_DEVICE != 'mps' else 'cpu'   # MPS may not support all Kronos ops
        predictor = KronosPredictor(kmodel, tokenizer, device=kronos_device, max_context=512)
        print(f'Kronos loaded in {time.time()-t0:.0f}s (device={kronos_device})')

        CONTEXT_LEN  = 60
        PRED_HORIZON = 5
        test_dates    = sorted(X_test.index.get_level_values('timestamp').unique())
        strided_dates = test_dates[::KRONOS_STRIDE]
        print(f'Forecasting {len(TICKERS)} tickers x {len(strided_dates)} dates (stride={KRONOS_STRIDE})...')

        needed = {'price', 'high', 'low', 'volume'}
        avail  = [c for c in ['price', 'high', 'low', 'volume'] if c in master.columns]
        if 'price' not in avail:
            raise RuntimeError(f'Need price column in master, got: {avail}')
        print(f'Using OHLCV columns: {avail}')

        kronos_predictions = {}
        t0 = time.time()
        for i, ticker in enumerate(TICKERS):
            try:
                ticker_data = master.xs(ticker, level='ticker').sort_index()
            except KeyError:
                continue
            for date in strided_dates:
                try:
                    end_loc = ticker_data.index.get_loc(date)
                    ctx_raw = ticker_data.iloc[max(0, end_loc - CONTEXT_LEN):end_loc][avail].dropna()
                    if len(ctx_raw) < 20:
                        continue
                    close = ctx_raw['price']
                    ctx_ohlcv = pd.DataFrame({
                        'open':   close.shift(1).fillna(close),
                        'high':   ctx_raw['high']   if 'high'   in avail else close,
                        'low':    ctx_raw['low']    if 'low'    in avail else close,
                        'close':  close,
                        'volume': ctx_raw['volume'] if 'volume' in avail else pd.Series(1.0, index=close.index),
                    })
                    y_ts = pd.bdate_range(ctx_ohlcv.index[-1], periods=PRED_HORIZON + 1)[1:]
                    out = predictor.predict(
                        df=ctx_ohlcv,
                        x_timestamp=pd.Series(ctx_ohlcv.index),
                        y_timestamp=pd.Series(y_ts),
                        pred_len=PRED_HORIZON, T=1.0, top_p=0.9, sample_count=1,
                    )
                    pred_close = out['close'].values
                    current = float(close.iloc[-1])
                    kronos_predictions[(date, ticker)] = (float(pred_close[-1]) - current) / current
                except Exception:
                    continue
            if (i + 1) % 5 == 0:
                print(f'  ...{i + 1}/{len(TICKERS)} tickers done ({time.time() - t0:.0f}s)')

        if kronos_predictions:
            s = pd.Series(kronos_predictions)
            s.index = pd.MultiIndex.from_tuples(s.index, names=['timestamp', 'ticker'])
            wide = s.unstack('ticker').sort_index().reindex(test_dates).ffill().fillna(0)
            long = wide.stack().rename('kronos_pred')
            long.index = long.index.set_names(['timestamp', 'ticker'])
            kronos_pred_test = long.reindex(X_test.index).fillna(0)
            print(f'\nKronos done in {time.time()-t0:.0f}s. Predictions: {(kronos_pred_test != 0).sum():,}')
            kronos_scores = score_model('Kronos', kronos_pred_test, y_test)
        else:
            print('No Kronos predictions captured')
    except Exception as e:
        print(f'Kronos run FAILED: {type(e).__name__}: {e}')

print(f'\nKronos scores on test:')
for k, v in kronos_scores.items():
    if k != 'model':
        print(f'  {k:18s} = {v}')

check('Kronos predictions made', (kronos_pred_test != 0).any(),
      f'{(kronos_pred_test != 0).sum():,} non-zero preds', critical=False)


## 8. Compare models — comparison table + chart

In [ ]:
results_df = pd.DataFrame([lgb_scores, chronos_scores, vqc_scores, xpyq_scores, timesfm_scores, kronos_scores]).set_index('model')
print('=' * 60)
print(' ALPHA MODEL COMPARISON (test set)')
print('=' * 60)
display(results_df.round(4))

results_df.to_csv(os.path.join(DATA_DIR, 'stage2_model_comparison.csv'))

# Bar chart
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, metric in zip(axes, ['IC', 'dir_acc', 'long_short_sharpe']):
    vals = results_df[metric].fillna(0)
    colors = ['#4caf50' if 'VQC' in m else '#2196f3' for m in vals.index]
    ax.bar(vals.index, vals.values, color=colors, alpha=0.8)
    ax.set_title(metric)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.grid(alpha=0.3, axis='y')
    if metric == 'IC':
        ax.axhline(0.05, color='red', linestyle='--', alpha=0.5, label='IC=0.05 competitive')
        ax.legend(fontsize=8)
    elif metric == 'dir_acc':
        ax.axhline(0.57, color='red', linestyle='--', alpha=0.5, label='57% target')
        ax.legend(fontsize=8)
    elif metric == 'long_short_sharpe':
        ax.axhline(1.0, color='red', linestyle='--', alpha=0.5, label='Sharpe=1.0 target')
        ax.legend(fontsize=8)
plt.suptitle('Stage 2: Alpha model comparison vs hackathon targets', fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Pick primary model + save predictions

Best model becomes `primary`. All 3 saved for Stage 3 ablation and Stage 6 comparison table.

In [ ]:
# Pick primary by absolute IC (most predictive in either direction)
valid_ic = results_df['IC'].dropna()
if len(valid_ic) == 0:
    primary_name = 'LightGBM'
    print('WARN: no valid IC, defaulting primary to LightGBM')
else:
    primary_name = valid_ic.abs().idxmax()
print(f'PRIMARY alpha model: {primary_name}  (IC = {results_df.loc[primary_name, "IC"]:.4f})')

pred_lookup = {
    'LightGBM':  lgb_pred_test,
    'Chronos':   chronos_pred_test,
    'VQC':       vqc_pred_test,
    'XpyQ-EWMA': xpyq_pred_test,
    'TimesFM':   timesfm_pred_test,
    'Kronos':    kronos_pred_test,
}

# Save all three + primary, indexed by (timestamp, ticker)
alpha_mu = pd.DataFrame({
    'lightgbm': lgb_pred_test,
    'chronos':  chronos_pred_test,
    'vqc':      vqc_pred_test,
    'xpyq':     xpyq_pred_test,
    'timesfm':  timesfm_pred_test,
    'kronos':   kronos_pred_test,
    'primary':  pred_lookup[primary_name],
})
alpha_mu.to_parquet(os.path.join(DATA_DIR, 'alpha_mu.parquet'))
print(f'\nSaved alpha_mu.parquet: {alpha_mu.shape}')
print(f'  -> {DATA_DIR}/alpha_mu.parquet')

# Persist metadata about primary pick
with open(os.path.join(DATA_DIR, 'stage2_primary.json'), 'w') as f:
    json.dump({
        'primary_model': primary_name,
        'IC': float(results_df.loc[primary_name, 'IC']),
        'dir_acc': float(results_df.loc[primary_name, 'dir_acc']) if not pd.isna(results_df.loc[primary_name, 'dir_acc']) else None,
        'sharpe': float(results_df.loc[primary_name, 'long_short_sharpe']) if not pd.isna(results_df.loc[primary_name, 'long_short_sharpe']) else None,
        'all_results': results_df.to_dict(),
    }, f, indent=2, default=str)


# ===== Save trained model artifacts for inference (03_inference.ipynb) =====
import pickle
MODELS_DIR = os.path.join(WORKSPACE, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

# LightGBM: native save
lgb_model.booster_.save_model(os.path.join(MODELS_DIR, 'lgb_alpha.txt'))
print(f'Saved LightGBM -> {MODELS_DIR}/lgb_alpha.txt')

# VQC: torch state_dict + scaler + feature names
torch.save(vqc_model.state_dict(), os.path.join(MODELS_DIR, 'vqc_alpha.pt'))
with open(os.path.join(MODELS_DIR, 'vqc_scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)
with open(os.path.join(MODELS_DIR, 'vqc_features.json'), 'w') as f:
    json.dump({'features': top4, 'n_qubits': N_QUBITS, 'n_layers': N_LAYERS}, f)
print(f'Saved VQC -> {MODELS_DIR}/vqc_alpha.pt (+ scaler + features)')

# Save the feature column list so inference uses the same schema
with open(os.path.join(MODELS_DIR, 'signal_cols.json'), 'w') as f:
    json.dump({'signal_cols': SIGNAL_COLS, 'target': TARGET, 'tickers': TICKERS}, f)
print(f'Saved schema -> {MODELS_DIR}/signal_cols.json')

# Sample preview
print('\nSample predictions (last 5 test days, AAPL):')
display(alpha_mu.xs('AAPL', level='ticker').tail(5))

## 10. Stage 3 readiness summary

In [26]:
print('=' * 72)
print(' STAGE 2 READINESS SUMMARY')
print('=' * 72)

print('\nModel performance (test set):')
for name, row in results_df.iterrows():
    ic = row['IC']
    da = row['dir_acc']
    sh = row['long_short_sharpe']
    print(f'  {name:12s}: IC={ic:+.4f}  dir_acc={da:.3f}  long_short_Sharpe={sh:.3f}')

print(f'\nPrimary model: {primary_name}')
print(f'  Predictions saved to alpha_mu.parquet (columns: lightgbm, chronos, vqc, primary)')

# Hackathon target comparison
print('\nVs hackathon targets:')
p_ic = results_df.loc[primary_name, 'IC']
p_da = results_df.loc[primary_name, 'dir_acc']
p_sh = results_df.loc[primary_name, 'long_short_sharpe']
print(f'  IC > 0.05 competitive:    {p_ic:.4f}  {"PASS" if abs(p_ic) > 0.05 else "BELOW"}')
print(f'  Directional accuracy >57%: {p_da:.3f}  {"PASS" if p_da > 0.57 else "BELOW"}')
print(f'  Long-short Sharpe > 1.0:   {p_sh:.3f}  {"PASS" if p_sh > 1.0 else "BELOW"}')

all_critical_fail = [n for n, (s, _) in CHECKS.items() if s == 'FAIL']
print()
print('=' * 72)
if all_critical_fail:
    print(f' OVERALL: RED - {len(all_critical_fail)} critical check(s) failed')
    for n in all_critical_fail:
        print(f'   - {n}: {CHECKS[n][1]}')
else:
    print(' OVERALL: GREEN - ready for Stage 3 (QSVM meta-labeling)')
    print('   Next: notebook 03_meta_labeling.ipynb loads alpha_mu.parquet + meta_features.parquet')
print('=' * 72)

with open(os.path.join(DATA_DIR, 'stage2_checks.json'), 'w') as f:
    json.dump({n: {'status': s, 'msg': m} for n, (s, m) in CHECKS.items()}, f, indent=2)

 STAGE 2 READINESS SUMMARY

Model performance (test set):
  LightGBM    : IC=+0.0841  dir_acc=0.552  long_short_Sharpe=1.012
  Chronos     : IC=+nan  dir_acc=0.000  long_short_Sharpe=0.523
  VQC         : IC=-0.0161  dir_acc=0.512  long_short_Sharpe=0.258
  XpyQ-EWMA   : IC=+nan  dir_acc=nan  long_short_Sharpe=nan

Primary model: LightGBM
  Predictions saved to alpha_mu.parquet (columns: lightgbm, chronos, vqc, primary)

Vs hackathon targets:
  IC > 0.05 competitive:    0.0841  PASS
  Directional accuracy >57%: 0.552  BELOW
  Long-short Sharpe > 1.0:   1.012  PASS

 OVERALL: GREEN - ready for Stage 3 (QSVM meta-labeling)
   Next: notebook 03_meta_labeling.ipynb loads alpha_mu.parquet + meta_features.parquet
